# Hospital Readmission Prediction

This notebook builds a simple **logistic regression** model to predict whether a patient will be readmitted within 30 days.

The example uses a small synthetic dataset so that the notebook can run immediately. Replace it with a real hospital dataset for your case study.

**Important:** This is an educational example, not a clinically validated model.

## 1. Import libraries

We use:
- `pandas` for handling tabular data
- `scikit-learn` for preprocessing, model training, and evaluation
- `matplotlib` for plotting the ROC curve

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)


## 2. Create a simple example dataset

Each row represents one patient.

The target column `readmitted_30_days` is:
- `1`: readmitted within 30 days
- `0`: not readmitted within 30 days

In a real project, load your CSV file instead.

In [ ]:
np.random.seed(42)
n_patients = 500

data = pd.DataFrame({
    'age': np.random.randint(18, 90, n_patients),
    'previous_visits': np.random.poisson(2, n_patients),
    'heart_rate': np.random.normal(80, 12, n_patients).round(1),
    'systolic_bp': np.random.normal(130, 18, n_patients).round(1),
    'diagnosis': np.random.choice(
        ['Diabetes', 'Heart Disease', 'Infection', 'Injury'],
        n_patients
    )
})

# Create a simple probability of readmission for demonstration.
# This is artificial data and has no medical meaning.
risk_score = (
    -3.0
    + 0.025 * data['age']
    + 0.30 * data['previous_visits']
    + 0.35 * (data['diagnosis'] == 'Heart Disease')
    + 0.20 * (data['diagnosis'] == 'Diabetes')
)

probability = 1 / (1 + np.exp(-risk_score))
data['readmitted_30_days'] = np.random.binomial(1, probability)

data.head()


## 3. Inspect the data

Before training, check the shape, data types, and missing values.

In [ ]:
print('Dataset shape:', data.shape)
print('\nData types:')
print(data.dtypes)
print('\nMissing values:')
print(data.isnull().sum())
print('\nTarget distribution:')
print(data['readmitted_30_days'].value_counts())


## 4. Separate features and target

`X` contains the patient information used for prediction.
`y` contains the answer we want the model to learn.

In [ ]:
X = data.drop(columns=['readmitted_30_days'])
y = data['readmitted_30_days']

numeric_features = [
    'age',
    'previous_visits',
    'heart_rate',
    'systolic_bp'
]

categorical_features = ['diagnosis']


## 5. Split into training and testing data

The model learns from the training set and is evaluated on unseen test data.

`stratify=y` keeps the proportion of readmitted and non-readmitted patients approximately similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))


## 6. Build the preprocessing pipeline

Numerical features are standardized so that their scales are comparable.
Categorical features are converted into numerical columns using one-hot encoding.

The preprocessing is placed inside a pipeline to avoid data leakage from the test set.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])


## 7. Create logistic regression with L2 regularization

`penalty='l2'` applies L2 regularization.

`C` is the inverse of regularization strength:
- Smaller `C` → stronger regularization
- Larger `C` → weaker regularization

The model will output probabilities, which are needed for ROC-AUC.

In [ ]:
model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(
        penalty='l2',
        C=1.0,
        max_iter=1000,
        random_state=42
    ))
])

# Train the complete pipeline.
model.fit(X_train, y_train)


## 8. Generate predictions

We obtain:
- Probabilities using `predict_proba()`
- Class predictions using `predict()`

ROC-AUC should be calculated using probabilities, not only 0/1 predictions.

In [ ]:
y_probability = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print('First five predicted probabilities:', y_probability[:5])
print('First five class predictions:', y_pred[:5])


## 9. Evaluate using ROC-AUC

ROC-AUC measures how well the model ranks patients who are readmitted above patients who are not readmitted.

A value near 0.5 is similar to random ranking. Higher values indicate better discrimination.

In [ ]:
auc_score = roc_auc_score(y_test, y_probability)
print(f'ROC-AUC: {auc_score:.3f}')

false_positive_rate, true_positive_rate, thresholds = roc_curve(
    y_test,
    y_probability
)

plt.figure(figsize=(7, 5))
plt.plot(false_positive_rate, true_positive_rate, label=f'ROC-AUC = {auc_score:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate / Recall')
plt.title('ROC Curve')
plt.legend()
plt.grid(True)
plt.show()


## 10. Confusion matrix and classification report

A confusion matrix shows:
- True Positives: correctly predicted readmissions
- True Negatives: correctly predicted non-readmissions
- False Positives: predicted readmission, but no readmission occurred
- False Negatives: missed readmissions


In [ ]:
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['No readmission', 'Readmission']
)
display.plot()
plt.title('Confusion Matrix')
plt.show()


## 11. Clinical interpretation of errors

### False negative
The patient is actually readmitted within 30 days, but the model predicts no readmission. This may result in missed follow-up or preventive care.

### False positive
The model predicts readmission, but the patient is not readmitted. This may cause unnecessary follow-up, extra workload, or additional costs.

In many settings, false negatives may be more costly than false positives. However, the correct trade-off depends on the clinical intervention, available resources, and patient safety requirements.

A lower decision threshold can identify more high-risk patients, but it usually increases false positives.

## 12. Try a different classification threshold

The default threshold is usually 0.5. Here we try 0.3 to identify more potential readmissions.

This changes the class predictions, but it does not change the ROC-AUC because ROC-AUC uses the continuous probabilities.

In [ ]:
threshold = 0.30
y_pred_custom = (y_probability >= threshold).astype(int)

print(f'Classification threshold: {threshold}')
print(classification_report(y_test, y_pred_custom, zero_division=0))

cm_custom = confusion_matrix(y_test, y_pred_custom)
ConfusionMatrixDisplay(
    confusion_matrix=cm_custom,
    display_labels=['No readmission', 'Readmission']
).plot()
plt.title(f'Confusion Matrix at Threshold {threshold}')
plt.show()


## 13. What to change for your real case study

1. Replace the synthetic dataset with a real, ethically sourced dataset.
2. Identify the exact target column for 30-day readmission.
3. Remove data leakage: use only information available before the prediction time.
4. Handle diagnosis codes appropriately.
5. Check class imbalance.
6. Report ROC-AUC, confusion matrix, precision, recall, and false-negative count.
7. Explain the clinical cost of false negatives versus false positives.
8. Treat the result as a research model unless it has been properly clinically validated.